# 00 — Prepare a causal teaching dataset

This notebook is the first step in the tutorial.

It starts from the code/docstring corpus and produces a **unit-level observational dataset** with deliberately different causal roles. That gives notebooks 01 and 02 a richer example than a collection of highly correlated code-size measurements.

### Tutorial path

**00 Data preparation → 01 Correlational analysis → 02 Causal inference**

The causal question for the synthetic example is:

> **Does enabling a documentation intervention improve the probability of a successful documentation outcome?**

The original code and docstring text are left unchanged. Three synthetic baseline/context fields were added to the raw CSV so the example can contain an instrument-like variable, a genuine confounder, and an irrelevant variable.


## 1. Configure the preparation

The bundle uses project-relative paths, so it should run after you open Jupyter from the project directory.

The source file contains:

- the original `input_code`;
- the original `output_docstring`;
- synthetic `developer_experience`;
- synthetic `rollout_eligibility`;
- synthetic `noise_feature`.

The final causal table will also contain post-treatment variables created by the synthetic data-generating process.


In [ ]:
from src.causal_data_prep import (
    DEFAULT_COVARIATES,
    engineer_features,
    load_source_data,
    make_synthetic_observational_data,
    save_causal_dataset,
    save_ground_truth_dag,
    validate_causal_dataset,
)

def default_params():
    return {
        "source_dataset": "data/raw_code.csv",
        "lizard_cache_folder": "cache/lizard",
        "causal_dataset": "data/causal_data.csv",
        "ground_truth_dag": "data/synthetic_ground_truth_edges.csv",
        "random_seed": 42,
        "covariate_columns": DEFAULT_COVARIATES,
    }

params = default_params()
params


## 2. Load the enriched raw corpus

The added context variables are synthetic and exist only to make the causal example pedagogically useful.

They are all determined **before treatment**:

- `developer_experience`: a baseline factor that can affect treatment uptake and outcome;
- `rollout_eligibility`: an encouragement/availability signal that affects treatment uptake;
- `noise_feature`: unrelated background variation.

We do not modify the original code or docstring strings.


In [ ]:
source_df = load_source_data(params["source_dataset"])

print(f"Rows: {len(source_df):,}")
source_df[
    [
        "input_code",
        "output_docstring",
        "developer_experience",
        "rollout_eligibility",
        "noise_feature",
    ]
].head()


## 3. Extract baseline code features

We retain a few code measurements that have useful teaching behavior:

- `code_number_tokens`
- `code_complexity`
- `code_num_identifiers`
- `code_num_strings`

The reference docstring is also summarized internally so the synthetic generator can create a realistic post-treatment detail score.

Not every measured feature will become a cause in the ground-truth DAG. That is intentional: notebook 01 should contain correlated measurements, proxies, and irrelevant variables as well as true causes.


In [ ]:
feature_df = engineer_features(
    source_df,
    cache_dir=params["lizard_cache_folder"],
)

feature_df[
    [
        "code_number_tokens",
        "code_complexity",
        "code_num_identifiers",
        "code_num_strings",
        "reference_docstring_words",
        "developer_experience",
        "rollout_eligibility",
        "noise_feature",
    ]
].describe().T


## 4. Generate the synthetic observational study

The generator creates one observed treatment and one observed outcome per unit.

It also creates two post-treatment variables:

- `docstring_detail_score`: affected by treatment and able to affect outcome;
- `review_flag`: affected by both treatment and the realized outcome.

Those two variables make the later DAG exercise more interesting because a strong correlation with treatment and outcome does not imply “confounder.”

The exact ground-truth DAG is saved for a reveal exercise in notebook 02. Try not to inspect it yet if you want to do the causal-graph exercise yourself.


In [ ]:
causal_df, study_info = make_synthetic_observational_data(
    feature_df,
    seed=params["random_seed"],
)

print(f"Treatment prevalence: {study_info.treatment_prevalence:.3f}")
print(f"Outcome prevalence:   {study_info.outcome_prevalence:.3f}")
print(
    "Known synthetic total ATE: "
    f"{study_info.true_average_treatment_effect:.3f}"
)

causal_df.head()


## 5. Validate the causal table

The downstream notebooks require:

- one row per unit;
- both treatment groups;
- a binary outcome;
- numeric finite covariates;
- no missing analysis values.


In [ ]:
validation_summary = validate_causal_dataset(
    causal_df,
    covariates=params["covariate_columns"],
)

validation_summary


## 6. Save the handoff files

`causal_data.csv` is consumed by notebooks 01 and 02.

The ground-truth edge file is for the **final reveal** in notebook 02. Notebook 01 deliberately does not use it.


In [ ]:
data_path = save_causal_dataset(
    causal_df,
    params["causal_dataset"],
)
truth_path = save_ground_truth_dag(
    params["ground_truth_dag"],
)

print(f"Saved causal data: {data_path}")
print(f"Saved hidden teaching DAG: {truth_path}")


## Next: 01 — Correlational analysis

Notebook 00 has defined what was observed.

Notebook 01 now hides the data-generating story and asks only what can be learned from the observed table:

- Which variables differ across treatment groups?
- Which are associated with outcome?
- Which appear related to both?
- Which variables might be mediators, colliders, instruments, proxies, or irrelevant?

The goal is to experience firsthand why those roles cannot be read directly from a correlation matrix.
